In [ ]:
import numpy as np
from pathlib import Path

from brainvision.constants import *
from brainvision.data import load_patient, list_patient_zips
from brainvision.preprocessing import preprocess       # ← replaces all five functions

# All five pipeline functions (calibrate, smooth_spectra, remove_noisy_bands,
# decimate_spectral_channels, minmax_normalise) are now available through
# brainvision.preprocessing if needed individually.

## Pre-processing

In [ ]:
def save_processed_patient(patient: dict,
                            out_dir: str) -> Path:
    """Save preprocessed cube and labels to a compressed .npz file."""
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    out_path = Path(out_dir) / f"{patient['id']}.npz"
    np.savez_compressed(
        out_path,
        processed = patient['processed'],   # (H, W, 128) float32
        labels    = patient['labels'],       # (H, W)      int32
    )
    return out_path

In [22]:
def load_processed_patient(npz_path: Path) -> dict:
    """Load a single preprocessed patient from a .npz file."""
    data = np.load(npz_path)
    return {
        'id'        : npz_path.stem,
        'processed' : data['processed'],    # (H, W, 128) float32
        'labels'    : data['labels'],        # (H, W)      int32
    }

In [23]:
def load_processed_patients(data_dir: str = PROCESSED_DIRS[1]) -> list[dict]:
    """
    Load all preprocessed patients from disk.
    Used at the top of 03_training.ipynb and 04_evaluation.ipynb.
    """
    paths    = sorted(Path(data_dir).glob("*.npz"))
    patients = [load_processed_patient(p) for p in paths]
    print(f"Loaded {len(patients)} preprocessed patients from {data_dir}")
    return patients

In [24]:
def preprocess_all(data_dir:  str = CAMPAIGN_DIRS[1],
                   out_dir:   str = PROCESSED_DIRS[1],
                   overwrite: bool = False):
    """
    Load, preprocess, and save all patient ZIPs to disk.

    Args:
        data_dir  : folder containing raw .zip files
        out_dir   : folder to save .npz files
        overwrite : if False, skip patients already saved to disk
    """
    all_zips = list_patient_zips(data_dir)
    print(f"  Found {len(all_zips)} ZIPs in {data_dir}")
    print(f"  Saving to {out_dir}\n")

    saved, skipped = 0, 0

    for zip_path in all_zips:
        out_path = Path(out_dir) / f"{zip_path.stem}.npz"

        if out_path.exists() and not overwrite:
            print(f"  Skipping {zip_path.stem} — already exists")
            skipped += 1
            continue

        patient  = load_patient(str(zip_path), cleanup=True)
        patient  = preprocess(patient)
        save_processed_patient(patient, out_dir)
        del patient # Free memory before next patient
        saved += 1

    print(f"\n{'─'*50}")
    print(f"  Saved   : {saved}")
    print(f"  Skipped : {skipped}")
    print(f"  Total   : {saved + skipped} / {len(all_zips)}")
    print(f"{'─'*50}")


In [25]:
def verify_processed(data_dir: str):
    """
    Confirm every .npz has the expected shape, dtype, and [0, 1] range.
    Raises AssertionError immediately if anything is wrong.
    """
    paths  = sorted(Path(data_dir).glob("*.npz"))
    errors = []

    print(f"\n  {'ID':<12} {'Shape':<20} {'Labels':<15} "
          f"{'Min':>6} {'Max':>6} {'dtype':<10} {'OK?'}")
    print(f"  {'─'*75}")

    for p in paths:
        d         = np.load(p)
        proc      = d['processed']
        labels    = d['labels']
        ok        = True
        reasons   = []

        if proc.ndim != 3 or proc.shape[2] != N_DECIMATED_BANDS:
            ok = False
            reasons.append(f"shape={proc.shape}")
        if proc.dtype != np.float32:
            ok = False
            reasons.append(f"dtype={proc.dtype}")
        if proc.min() < -1e-4 or proc.max() > 1.0001:
            ok = False
            reasons.append(f"range=[{proc.min():.4f},{proc.max():.4f}]")

        status = "✅" if ok else f"❌ {', '.join(reasons)}"
        print(f"  {p.stem:<12} {str(proc.shape):<20} "
              f"{str(labels.shape):<15} "
              f"{proc.min():>6.3f} {proc.max():>6.3f} "
              f"{str(proc.dtype):<10} {status}")

        if not ok:
            errors.append(p.stem)

    print(f"\n  {'─'*75}")
    if errors:
        print(f"  ❌ {len(errors)} files failed: {errors}")
    else:
        print(f"  ✅ All {len(paths)} files verified")

In [26]:
for campaign_id in [1, 2, 3]:
    print(f"\n{'═'*55}")
    print(f"  Campaign {campaign_id}")
    print(f"{'═'*55}")
    preprocess_all(
        data_dir  = CAMPAIGN_DIRS[campaign_id],
        out_dir   = PROCESSED_DIRS[campaign_id],
        overwrite = False,
    )
    verify_processed(PROCESSED_DIRS[campaign_id])

print(f"\n{'═'*55}")
print(f"  Preprocessing complete — all campaigns saved")
print(f"{'═'*55}")


═══════════════════════════════════════════════════════
  Campaign 1
═══════════════════════════════════════════════════════
  Found 27 ZIPs in ../datasets/first_campaign
  Saving to ../processed/first_campaign

  Skipping 004-02 — already exists
  Skipping 005-01 — already exists
  Skipping 007-01 — already exists
  Skipping 008-01 — already exists
  Skipping 008-02 — already exists
  Skipping 010-03 — already exists
  Skipping 012-01 — already exists
  Skipping 012-02 — already exists
  Skipping 013-01 — already exists
  Skipping 014-01 — already exists
  Skipping 015-01 — already exists
  Skipping 016-01 — already exists
  Skipping 016-02 — already exists
  Skipping 016-03 — already exists
  Skipping 016-04 — already exists
  Skipping 016-05 — already exists
  Skipping 017-01 — already exists
  Skipping 018-01 — already exists
  Skipping 018-02 — already exists
  Skipping 019-01 — already exists
  Skipping 020-01 — already exists
  Skipping 021-01 — already exists
  Skipping 021-02